# 最佳融合模型完整指标（便于与 AcrPred 全面对比）

**目的**：复现 ProteinBERT + PSSM 融合中表现最好的配置（**ProteinBERT_PSSM1110**），在相同 test set 上计算**完整评估指标**，便于与 AcrPred (IJBM 2023) 做全面对比。

**与 AcrPred 论文的对应关系**：
- 数据来源与划分与 AcrPred 一致（anti-CRISPRdb + 统一资源，CD-HIT 去冗余，同一 train/test 划分）。
- AcrPred 原文仅报告 **AUC, ACC, SN (Sensitivity), SP (Specificity)**；本 notebook 额外输出 **AUPRC, F1, MCC, Brier, ECE** 及验证集最优阈值，便于多维度对比与撰写论文。

**参考**：`anticrispr_demo.ipynb`（数据加载、FusionTrainConfig、训练流程）；`confusion_matrix_demo.ipynb`（Fusion_PSSM1110 训练与评估）。

## 1. 依赖与路径

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from tensorflow import keras

from proteinbert import (
    load_anticrispr_with_ids,
    load_pretrained_model,
    FusionTrainConfig,
    load_feature_cache,
    attach_pssm_features,
)
from proteinbert.pssm_fusion import (
    _build_late_fusion_model,
    _encode_x,
    expected_calibration_error,
    find_best_threshold,
)

PROJECT_ROOT = '/home/nemophila/projects/protein_bert'
BENCHMARKS_DIR = f'{PROJECT_ROOT}/anticrispr_benchmarks'
WORK_ROOT = os.environ.get('PSSM_WORK_ROOT', '/home/nemophila/data/pssm_work')
FEAT_DIR = f'{WORK_ROOT}/features'

# 与 full experiment / confusion_matrix_demo 一致，固定种子便于复现与对比
SEED = 22
PSSM_VARIANT = '1110'

2026-03-07 15:09:44.766286: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


## 2. 扩展评估函数（含 ACC, SN, SP，与 AcrPred 论文指标对齐）

In [2]:
def evaluate_binary_full(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    """
    二分类完整指标，包含 AcrPred 论文中的 ACC/SN/SP 以及 AUPRC/F1/MCC/Brier/ECE。
    正类为 Acr (label=1)，SN = Sensitivity = Recall for positive, SP = Specificity = TN/(TN+FP)。
    """
    y_cls = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_cls).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    acc = accuracy_score(y_true, y_cls)
    return {
        'AUC': float(roc_auc_score(y_true, y_prob)),
        'AUPRC': float(average_precision_score(y_true, y_prob)),
        'F1': float(f1_score(y_true, y_cls)),
        'MCC': float(matthews_corrcoef(y_true, y_cls)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'ECE': float(expected_calibration_error(y_true, y_prob, n_bins=10)),
        'ACC': float(acc),
        'SN': float(sn),
        'SP': float(sp),
        'Threshold': float(threshold),
    }

## 3. 加载数据（1110 维 PSSM，与 anticrispr_demo 一致）

In [3]:
train_base_df, test_base_df = load_anticrispr_with_ids(BENCHMARKS_DIR, benchmark_name='anticrispr_binary')
parquet_path = f'{FEAT_DIR}/pssm_features_{PSSM_VARIANT}.parquet'
csv_path = f'{FEAT_DIR}/pssm_features_{PSSM_VARIANT}.csv'
cache_path = parquet_path if os.path.exists(parquet_path) else csv_path
if not os.path.exists(cache_path):
    raise FileNotFoundError(f'PSSM cache not found: {cache_path}')

feature_df, feature_cols = load_feature_cache(cache_path)
train_df = attach_pssm_features(train_base_df, feature_df, feature_cols)
test_df = attach_pssm_features(test_base_df, feature_df, feature_cols)
y_test = test_df['label'].astype(int).to_numpy()

print('train:', train_df.shape, 'test:', test_df.shape, 'PSSM dim:', len(feature_cols))

train: (1107, 1113) test: (286, 1113) PSSM dim: 1110


## 4. 训练 ProteinBERT + PSSM1110 融合模型（与 anticrispr_demo 相同配置）

In [4]:
cfg = FusionTrainConfig(
    seq_len=512,
    batch_size=8,
    frozen_epochs=6,
    unfrozen_epochs=12,
    frozen_lr=1e-4,
    unfrozen_lr=2e-5,
    pssm_dropout=0.3,
    global_dropout=0.3,
    pssm_hidden_dim=128,
    global_hidden_dim=128,
    global_bottleneck_dim=64,
    fusion_hidden_dim=128,
    use_hidden_global_concat=True,
)

rng_train, rng_valid = train_test_split(
    train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED
)

x_train = rng_train[feature_cols].to_numpy(dtype=np.float32)
x_valid = rng_valid[feature_cols].to_numpy(dtype=np.float32)
x_test = test_df[feature_cols].to_numpy(dtype=np.float32)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_valid = scaler.transform(x_valid)
x_test = scaler.transform(x_test)

y_train = rng_train['label'].astype(int).to_numpy()
y_valid = rng_valid['label'].astype(int).to_numpy()

pmg, enc = load_pretrained_model(
    local_model_dump_dir=f'{PROJECT_ROOT}/proteinbert_models',
    download_model_dump_if_not_exists=True,
    validate_downloading=False,
)

X_train = _encode_x(enc, rng_train['seq'].tolist(), cfg.seq_len, x_train)
X_valid = _encode_x(enc, rng_valid['seq'].tolist(), cfg.seq_len, x_valid)
X_test = _encode_x(enc, test_df['seq'].tolist(), cfg.seq_len, x_test)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=cfg.patience, restore_best_weights=True
    )
]

model = _build_late_fusion_model(
    pmg, seq_len=cfg.seq_len, pssm_dim=len(feature_cols),
    freeze_pretrained_layers=True, cfg=cfg,
)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.frozen_lr), loss='binary_crossentropy')
model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
          epochs=cfg.frozen_epochs, batch_size=cfg.batch_size, callbacks=callbacks, verbose=1)

for layer in model.layers:
    layer.trainable = True
model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.unfrozen_lr), loss='binary_crossentropy')
model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
          epochs=cfg.unfrozen_epochs, batch_size=cfg.batch_size, callbacks=callbacks, verbose=1)

valid_prob = model.predict(X_valid, batch_size=cfg.batch_size, verbose=0).reshape(-1)
thr = find_best_threshold(y_valid, valid_prob)
test_prob = model.predict(X_test, batch_size=cfg.batch_size, verbose=0).reshape(-1)

print('Best threshold (valid F1):', thr)

2026-03-07 15:09:46.418740: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-03-07 15:09:46.419734: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-03-07 15:09:46.448746: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:2a:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-03-07 15:09:46.448894: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 1 with properties: 
pciBusID: 0000:ab:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-03-07 15:09:46.448915: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-03-07 15:09:46.4

Epoch 1/6


2026-03-07 15:09:54.479198: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-03-07 15:09:55.242288: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-03-07 15:09:55.253293: I tensorflow/stream_executor/cuda/cuda_blas.cc:1838] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-03-07 15:09:55.254605: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.8
2026-03-07 15:09:57.450065: W tensorflow/stream_executor/gpu/asm_compiler.cc:63] Running ptxas --version returned 256
2026-03-07 15:09:57.630677: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] Internal: ptxas exited with non-zero error code 256, output: 
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.


125/125 [==============================] - 33s 150ms/step - loss: 0.4896 - val_loss: 0.2874
Epoch 2/6
125/125 [==============================] - 3s 26ms/step - loss: 0.3591 - val_loss: 0.2332
Epoch 3/6
125/125 [==============================] - 3s 26ms/step - loss: 0.2754 - val_loss: 0.2276
Epoch 4/6
125/125 [==============================] - 3s 25ms/step - loss: 0.2270 - val_loss: 0.2058
Epoch 5/6
125/125 [==============================] - 3s 26ms/step - loss: 0.2208 - val_loss: 0.1986
Epoch 6/6
125/125 [==============================] - 3s 26ms/step - loss: 0.1499 - val_loss: 0.1728
Epoch 1/12
125/125 [==============================] - 36s 152ms/step - loss: 0.1089 - val_loss: 0.1488
Epoch 2/12
125/125 [==============================] - 6s 47ms/step - loss: 0.1012 - val_loss: 0.1746
Epoch 3/12
125/125 [==============================] - 6s 47ms/step - loss: 0.0957 - val_loss: 0.1609
Epoch 4/12
125/125 [==============================] - 6s 47ms/step - loss: 0.0591 - val_loss: 0.1606
Ep

## 5. 完整指标与混淆矩阵

In [5]:
metrics = evaluate_binary_full(y_test, test_prob, threshold=thr)
for k, v in metrics.items():
    print(f'{k}: {v}')

y_pred = (test_prob >= thr).astype(int)
cm = confusion_matrix(y_test, y_pred)
print('\nConfusion matrix (test):')
print(cm)

AUC: 0.9355029585798817
AUPRC: 0.707136691740255
F1: 0.6176470588235294
MCC: 0.5903944222047595
Brier: 0.05593231243304165
ECE: 0.05474106036126613
ACC: 0.9090909090909091
SN: 0.8076923076923077
SP: 0.9192307692307692
Threshold: 0.3

Confusion matrix (test):
[[239  21]
 [  5  21]]


## 6. 与 AcrPred (IJBM 2023) 对比表

AcrPred 原文在**同一数据构建**的独立测试集上报告：AUC=0.952, SN=0.923, SP=0.877, ACC=0.881；未报告 AUPRC/F1/MCC/Brier/ECE。下表便于论文中直接引用。

In [6]:
acrpred_paper = {
    'AUC': 0.952,
    'ACC': 0.881,
    'SN': 0.923,
    'SP': 0.877,
    'AUPRC': None,
    'F1': None,
    'MCC': None,
    'Brier': None,
    'ECE': None,
}

ours = {
    'AUC': round(metrics['AUC'], 4),
    'ACC': round(metrics['ACC'], 4),
    'SN': round(metrics['SN'], 4),
    'SP': round(metrics['SP'], 4),
    'AUPRC': round(metrics['AUPRC'], 4),
    'F1': round(metrics['F1'], 4),
    'MCC': round(metrics['MCC'], 4),
    'Brier': round(metrics['Brier'], 4),
    'ECE': round(metrics['ECE'], 4),
}

comparison = pd.DataFrame({
    'AcrPred (IJBM 2023)': [acrpred_paper[k] if acrpred_paper[k] is not None else '—' for k in ours.keys()],
    'Ours (ProteinBERT+PSSM1110)': list(ours.values()),
}, index=list(ours.keys()))
comparison

,AcrPred (IJBM 2023),Ours (ProteinBERT+PSSM1110)
AUC,0.952,0.9355
ACC,0.881,0.9091
SN,0.923,0.8077
SP,0.877,0.9192
AUPRC,—,0.7071
F1,—,0.6176
MCC,—,0.5904
Brier,—,0.0559
ECE,—,0.0547
